# Tutorial 10: Poisson LASSO with Bike Sharing Data

## Learning goals

After working through this notebook, you should be able to:

1. Build a clean count-data set for Poisson regression from an existing data source.
2. Export that processed data set to a local CSV file.
3. Prepare a model matrix for `glmnet`.
4. Use `cv.glmnet()` to tune a **Poisson LASSO** model with **cross-validated deviance**.
5. Extract `lambda.1se` and fit the corresponding sparse model.
6. Identify which inputs are selected by the LASSO fit.


In [ ]:
# Run this cell before continuing.
library(tidyverse)
library(glmnet)
library(ISLR2)

source("tests_tutorial_10.R")

## 1. The dataset

In this lecture, we'll use the `Bikeshare`
[dataset](The UCI Machine Learning Repository https://archive.ics.uci.edu/ml/datasets/bike+sharing+dataset) from the `ISLR` package discussed in the [ISL book](https://www.statlearning.com)

The dataset contains information on the hourly users of a bike sharing program in Washington DC, for different months, hours, weather conditions, and membership types (see `?Bikeshare`).

Since the plain LASSO method is not particularly suited for categorical variables with many levels (group LASSO is), we will first collapse several categorical variables into binary indicators so that ordinary LASSO is more natural:

- `is_weekend` vs weekday
- `bad_weather` vs good weather
- `peak_season` vs off-season
- `rush_hour` vs non-rush

We also keep the numeric predictors in the dataset.

The response variable is the hourly count of riders, `bikers`.

*Run the cell below*

In [ ]:
data("Bikeshare", package = "ISLR2")

head(Bikeshare,2)

In [ ]:
# Run this cell to create the processed data set.

bike_poisson_lasso <- 
  Bikeshare |>
  transmute(
    # retain the response and numeric
    bikers = bikers,
    temperature = temp,
    feel_temp = atemp,
    humidity = hum,	
    windspeed = windspeed,	  
    # create categorical variables with only 2 levels
    holiday = factor(holiday),  
    is_weekend = factor(
      if_else(weekday %in% c(0, 6), "weekend", "weekday")
    ),
    bad_weather = factor(
  if_else(
    weathersit %in% c("light rain/snow", "heavy rain/snow"),
    "bad","good")
    ),
    peak_season = factor(
      if_else(as.integer(mnth) %in% 5:10, "peak", "off")
    ),
    rush_hour = factor(
      if_else(as.integer(hr) %in% c(7, 8, 9, 17, 18), "rush", "non_rush")
    )
  )

glimpse(bike_poisson_lasso)

In [ ]:
# Minimal EDA
summary(bike_poisson_lasso)
colSums(is.na(bike_poisson_lasso))

## 3. Split into training and test sets

We will use a 70/30 split, as in many of the other tutorial notebooks. Given that some categorical variables are not balanced, we'll make sure that proportions are maintained withing groups of these variables as well.

In [ ]:
set.seed(2026)

bike_train <- bike_poisson_lasso |>
  group_by(is_weekend, bad_weather, peak_season, rush_hour) |>
  sample_frac(0.70) |>
  ungroup()

bike_test <- bike_poisson_lasso |>
  anti_join(bike_train)

dim(bike_train)
dim(bike_test)

## 4. Prepare the model matrix for `glmnet`

As before, `glmnet` needs the inputs in matrix form.
We remove the intercept column from the matrix returned by `model.matrix()`.


**Question 1.0**  
<br>{points: 1}

Create a matrix with all predictors in `bike_train` using `model.matrix()` function. 

> Recall that the first column of the resulting matrix is a column of 1s to model the intercept, which we won't penalize.

Save the resulting matrix to an object called `model_matrix_X_train`.  

Also create the response matrix and save it as `matrix_Y_train`.

> Check Tutorial_09 if you don't remember how to do this.

*Fill out those parts indicated with `...`, uncomment the code, and run the cell.*

In [ ]:
# model_matrix_X_train <- 
#   ...(object = ..., data = ...)[, -1]

# matrix_Y_train <- 
#   ...(..., ncol = 1)

### BEGIN SOLUTION

model_matrix_X_train <- 
  model.matrix(object = bikers ~ ., data = bike_train)[, -1]

matrix_Y_train <- 
  as.matrix(bike_train$bikers, ncol = 1)

### END SOLUTION

dim(model_matrix_X_train)
dim(matrix_Y_train)

In [ ]:
head(model_matrix_X_train,1)

In [ ]:
source("tests_tutorial_10.R")

In [ ]:
test_1.0()

## 5. Building a Poisson-LASSO model

We now want to build a predictive Poisson model using a **LASSO** penalty. 

The steps are very similar to those needed to in MLR and Logistic regression with this penalty. However, there are 2 important differences:

- You need to adjust the `family` to be appropriate for a response with count data

- You need to choose an appropriate `type.measure` to select the level of penalization by cross-validation

**Question 1.1**  
<br>{points: 1}

We already prepared our training data with `model_matrix_X_train` and `matrix_Y_train`. Now we need to find an optimum value of $\lambda$ to build a **Poisson LASSO** model. 

> "Optimum" depends on the criteria we want to use to test performance. Check all possible `type.measure` for this family

Use the function `cv.glmnet()`, a 5-fold cross-validation and the deviance to test performance and select the level of penalization.

*Assign the output to `bike_cv_LASSO`*

In [ ]:
set.seed(1234) # do not change this!

# bike_cv_LASSO <- 
#   ...(
#     x = ...,
#     y = ...,
#     alpha = ...,
#     family = "...",
#     type.measure = "...",
#     nfolds = ...
#   )

# bike_cv_LASSO

### BEGIN SOLUTION

bike_cv_LASSO <- 
  cv.glmnet(
    x = model_matrix_X_train,
    y = matrix_Y_train,
    alpha = 1,
    family = "poisson",
    type.measure = "deviance",
    nfolds = 5
  )

### END SOLUTION

bike_cv_LASSO

In [ ]:
test_1.1()

### Selecting the level of penalization $\lambda$ 

The plot for `cv.glmnet()` shows the mean cross-validated deviance across the lambda grid, along with the dotted lines for `lambda.min` and `lambda.1se`.


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 8)

plot(
  bike_cv_LASSO,
  main = "Cross-Validation with Poisson LASSO\n\n"
)


**Question 1.2**  
<br>{points: 1}

Using `bike_cv_LASSO`, obtain $\lambda_{1se}$ and assign it to the variable `lambda_1se_deviance_LASSO`.

In [ ]:
# lambda_1se_deviance_LASSO <- ...

### BEGIN SOLUTION
lambda_1se_deviance_LASSO <- bike_cv_lambda_LASSO$lambda.1se
### END SOLUTION

lambda_1se_deviance_LASSO


In [ ]:
test_1.2()

**Question 1.3**  
<br>{points: 1}

Get the estimated coefficients of the **Poisson-LASSO** model corresponding to a level of penalization `lambda = lambda_1se_deviance_LASSO`.

*Assign the estimated coefficients to an object called `coefs_bike_LASSO`.*

In [ ]:
# coefs_bike_LASSO <- ...

### BEGIN SOLUTION

coefs_bike_LASSO <- coef(bike_cv_lambda_LASSO, s = "lambda.1se")

### END SOLUTION

coefs_bike_LASSO

In [ ]:
test_1.3()

**Question 1.4**  
<br>{points: 1}

Based on the coefficient output above, where coefficients equal to zero are shown as `.`, which input variables are selected by the Poisson-LASSO model built using the 1-SE rule with deviance as the performance metric?

**A.** `temperature`  
**B.** `feel_temp`  
**C.** `humidity`  
**D.** `windspeed`  
**E.** `holiday`  
**F.** `is_weekend`  
**G.** `bad_weather`  
**H.** `peak_season`  
**I.** `rush_hour`  

*Assign your answer to the object `answer1.4` as a single string containing the correct letters in **alphabetical order**. For example, if the selected variables were `temperature` and `holiday`, then you would write `"AE"`*

In [ ]:
# answer1.4 <- ""

### BEGIN SOLUTION
answer1.4 <- "ABCGI"
### END SOLUTION

answer1.4

In [ ]:
test_1.4()

## 6. Evaluate the selected model on the test set

The model selected by LASSO is considerably simpler than a full Poisson model and kept only 5 of the 9 possible predictors. We can now used the Poisson-LASSO model built on the trainind datat to predict number of bike rentals in the test set. 

**Question 1.5**  
<br>{points: 1}

Let's start by convering creating the matrices `model_matrix_X_test` and `model_matrix_y_test` from the test set.

*Fill out those parts indicated with ..., uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# model_matrix_X_test <- ...

# matrix_Y_test <- ...

### BEGIN SOLUTION
model_matrix_X_test <- 
    model.matrix(object = bikers ~ ., data = bike_test)[, -1]

matrix_Y_test <- 
  as.matrix(bike_test$bikers, ncol = 1)

### END SOLUTION

dim(model_matrix_X_test)
dim(matrix_Y_test)

In [ ]:
test_1.5()

**Question 1.6**  
<br>{points: 1}

Using the Poisson-LASSO model selected and estimated with the 1-SE rule, generate predicted bike rental counts for the testing set.

> Note that you can obtained this models from the object `bike_cv_lambda_LASSO` and an appropriate argument for `s`.

Store the predicted values in an object named `bike_pred_test`.

*Fill out those parts indicated with ..., uncomment the corresponding code in the cell below, and run it.*

In [ ]:
# bike_pred_test <- ...(
#     ..., 
#     newx = ...,
#     type = "...",
#     s = "...")

### BEGIN SOLUTION
bike_pred_test <- predict(
    bike_cv_lambda_LASSO, 
    newx = model_matrix_X_test,
    type = "response",
    s = "lambda.1se")
### END SOLUTION

head(bike_pred_test)

In [ ]:
test_1.6()